In [14]:
import pandas as pd

df = pd.read_csv("../data/processed/fomc_statements.csv", parse_dates=["date"])

# only the rate-decision markers. genuine policy statements set/hold the
# funds rate and almost always name it or its target range.
KEYWORDS = ["federal funds rate", "target range"]

text_lc = df["clean_text"].str.lower()
df["has_policy_kw"] = text_lc.apply(lambda t: any(k in t for k in KEYWORDS))
df["kw_hits"] = text_lc.apply(lambda t: ", ".join(k for k in KEYWORDS if k in t))

flagged = df.loc[~df["has_policy_kw"]]
print(f"flagged as NON-policy: {len(flagged)} of {len(df)}\n")
print(flagged[["date", "url"]].to_string())

flagged as NON-policy: 3 of 129

          date                                                                            url
77  2020-03-31  https://www.federalreserve.gov/newsevents/pressreleases/monetary20200331a.htm
81  2020-08-27  https://www.federalreserve.gov/newsevents/pressreleases/monetary20200827a.htm
122 2025-08-22  https://www.federalreserve.gov/newsevents/pressreleases/monetary20250822a.htm


In [18]:
flagged = df.loc[~df["has_policy_kw"]]

for _, row in flagged.iterrows():
    print(f"\n{'='*70}\n{row['date'].date()}  |  wc={len(row['clean_text'].split())}")
    print(f"url: {row['url']}")
    print(f"--- first 600 chars ---")
    print(row["clean_text"][:600])


2020-03-31  |  wc=249
url: https://www.federalreserve.gov/newsevents/pressreleases/monetary20200331a.htm
--- first 600 chars ---
The Federal Reserve on Tuesday announced the establishment of a temporary repurchase agreement facility for foreign and international monetary authorities (FIMA Repo Facility) to help support the smooth functioning of financial markets, including the U.S. Treasury market, and thus maintain the supply of credit to U.S. households and businesses. The FIMA Repo Facility will allow FIMA account holders, which consist of central banks and other international monetary authorities with accounts at the Federal Reserve Bank of New York, to enter into repurchase agreements with the Federal Reserve. In t

2020-08-27  |  wc=416
url: https://www.federalreserve.gov/newsevents/pressreleases/monetary20200827a.htm
--- first 600 chars ---
Following an extensive review that included numerous public events across the country, the Federal Open Market Committee (FOMC) on Thursday

In [19]:
# drop audited non-policy docs. all flagged docs were manually verified
# as non-policy (framework statement / repo facility / etc).
flagged = df.loc[~df["has_policy_kw"]]
policy = df.loc[df["has_policy_kw"]].copy().reset_index(drop=True)

print("before:", len(df), "-> after:", len(policy))
print("dropped:", len(flagged))

# confirm every flagged date is actually gone from the policy set — no hardcoding
dropped_dates = set(flagged["date"])
still_present = policy["date"].isin(dropped_dates).sum()
print("flagged docs still present in policy set:", still_present, "(should be 0)")

# quick look at the surviving set
print("\nclean policy set:", len(policy), "docs")
print("range:", policy["date"].min().date(), "->", policy["date"].max().date())

before: 129 -> after: 126
dropped: 3
flagged docs still present in policy set: 0 (should be 0)

clean policy set: 126 docs
range: 2011-01-26 -> 2026-04-29


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

MODEL = "ProsusAI/finbert"   # FinBERT fine-tuned for financial sentiment
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)
model.eval()   # inference mode — disables dropout, no gradient tracking needed

print("label map:", model.config.id2label)

# --- test on one statement ---
sample = policy["clean_text"].iloc[0]
print("\nsample date:", policy["date"].iloc[0].date(), "| words:", len(sample.split()))

# tokenize: truncation=True caps at the model's 512-token limit (long
# statements get cut; FOMC puts key policy language up front so front-truncation
# loses the least). return_tensors="pt" gives torch tensors.
inputs = tokenizer(sample, return_tensors="pt", truncation=True, max_length=512)

with torch.no_grad():                      # no gradients — inference only, faster + less memory
    logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=1)   # logits -> probabilities summing to 1

print("\nraw probs:", probs.tolist()[0])
print("label map again:", model.config.id2label)

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

label map: {0: 'positive', 1: 'negative', 2: 'neutral'}

sample date: 2011-01-26 | words: 390

raw probs: [0.3458772301673889, 0.30759215354919434, 0.34653061628341675]
label map again: {0: 'positive', 1: 'negative', 2: 'neutral'}


In [6]:
from tqdm import tqdm

def score_text(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        probs = torch.softmax(model(**inputs).logits, dim=1)[0]
    # label map confirmed: 0=positive, 1=negative, 2=neutral
    return probs[0].item(), probs[1].item(), probs[2].item()

# tqdm gives a progress bar — 127 docs on CPU takes a bit, this shows it's alive
results = [score_text(t) for t in tqdm(policy["clean_text"], desc="FinBERT")]

policy["prob_pos"] = [r[0] for r in results]
policy["prob_neg"] = [r[1] for r in results]
policy["prob_neu"] = [r[2] for r in results]

# compressed 0-centered score: +1 = fully positive, -1 = fully negative.
# 0-centered to match SPY returns -> clean sign interpretation in regression later.
policy["sentiment"] = policy["prob_pos"] - policy["prob_neg"]

# sanity check the distribution
print(policy[["prob_pos", "prob_neg", "prob_neu", "sentiment"]].describe())
print("\nmost positive:")
print(policy.nlargest(3, "sentiment")[["date", "sentiment", "prob_pos", "prob_neg"]].to_string())
print("\nmost negative:")
print(policy.nsmallest(3, "sentiment")[["date", "sentiment", "prob_pos", "prob_neg"]].to_string())

FinBERT:   0%|          | 0/126 [00:00<?, ?it/s]

FinBERT: 100%|██████████| 126/126 [02:12<00:00,  1.05s/it]

         prob_pos    prob_neg    prob_neu   sentiment
count  126.000000  126.000000  126.000000  126.000000
mean     0.343949    0.131608    0.524444    0.212341
std      0.215358    0.156976    0.224669    0.302597
min      0.032868    0.014200    0.087625   -0.842775
25%      0.141432    0.037236    0.326485    0.045108
50%      0.315479    0.075744    0.564465    0.232594
75%      0.488757    0.149751    0.714449    0.412616
max      0.887577    0.875643    0.900120    0.862778

most positive:
         date  sentiment  prob_pos  prob_neg
89 2021-11-03   0.862778  0.887577  0.024798
10 2012-04-25   0.766834  0.823443  0.056609
78 2020-06-10   0.734268  0.770704  0.036436

most negative:
         date  sentiment  prob_pos  prob_neg
65 2019-03-20  -0.842775  0.032868  0.875643
66 2019-05-01  -0.734961  0.042284  0.777245
68 2019-07-31  -0.679714  0.087004  0.766717


In [ ]:
out = policy[["date", "url", "clean_text",
              "prob_pos", "prob_neg", "prob_neu", "sentiment"]].copy()
out.to_csv("../data/processed/fomc_sentiment.csv", index=False)

check = pd.read_csv("../data/processed/fomc_sentiment.csv", parse_dates=["date"])
print("saved rows:", len(check))
print("date range:", check["date"].min().date(), "->", check["date"].max().date())
print("columns:", list(check.columns))
print("\nsentiment summary: mean={:.3f}  min={:.3f}  max={:.3f}".format(
    check["sentiment"].mean(), check["sentiment"].min(), check["sentiment"].max()))
print("\nhead:")
print(check[["date", "sentiment", "prob_pos", "prob_neg"]].head(3).to_string())

saved rows: 126
date range: 2011-01-26 -> 2026-04-29
columns: ['date', 'url', 'clean_text', 'prob_pos', 'prob_neg', 'prob_neu', 'sentiment']

sentiment summary: mean=0.212  min=-0.843  max=0.863

head:
        date  sentiment  prob_pos  prob_neg
0 2011-01-26   0.038285  0.345877  0.307592
1 2011-03-15   0.446380  0.488646  0.042266
2 2011-04-27   0.531847  0.563442  0.031595
